# Baseline Models on SUN RGB-D 19-Category

**Four fusion baselines using ResNet18 (0.75x width):**
1. RGB-only (single stream, 3-channel input)
2. Depth-only (single stream, 1-channel input)
3. Early fusion (RGB+Depth concatenated = 4-channel input)
4. Late fusion (two backbones, features concatenated before classifier)

All models use identical training configuration (optimizer, scheduler, grad clipping, label smoothing, dropout, epochs).
Mean class accuracy (MCA) is the primary metric.


## 1. Environment Setup & GPU Verification

In [1]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: Tesla T4
GPU Memory: 14.56 GB

T4 GPU detected - will be slower, consider upgrading to A100



In [2]:
# Detailed GPU info
!nvidia-smi

Fri Mar 27 14:34:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Mounted at /content/drive

Google Drive mounted successfully!

Drive contents:
total 3117893
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Jan 21  2023 activity (2).xlsx
-rw------- 1 root root        176 Jan 21  2023 activity.gsheet
-rw------- 1 

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [4]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

REPOSITORY SETUP
Cloning from https://github.com/clingergab/Multi-Stream-Neural-Networks.git...
Cloning into '/content/Multi-Stream-Neural-Networks'...
remote: Enumerating objects: 3200, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 3200 (delta 99), reused 99 (delta 85), pack-reused 3072 (from 2)
Receiving objects: 100% (3200/3200), 117.98 MiB | 17.51 MiB/s, done.
Resolving deltas: 100% (2000/2000), done.
Encountered 49 file(s) that should have been pointers, but weren't:
	tests/augmentation_comparison.png
	tests/augmentation_test.png
	tests/balanced_augmentation_test.png
	tests/balanced_samples_comparison.png
	tests/dataset_orthogonal_loading.png
	tests/decaying_restarts_eta_min_bug.png
	tests/easing_formula_analysis.png
	tests/easing_schedulers_comparison.png
	tests/global_vs_local_comparison.png
	tests/linear_scale_comparison.png
	tests/local_vs_global_orthogonal.png
	tests/log_vs_linear_scale_effect.png
	tests/m

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")
print(f"   thop: {thop.__version__}")

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 37.5 MB/s eta 0:00:00
All dependencies installed!
   h5py: 3.16.0
   matplotlib: 3.10.0
   kornia: 0.8.2


## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 19-category preprocessed (train + test splits, RGB + Depth)

In [6]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_traintest"  # Extracted location

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP (TRAIN + TEST)")
print("=" * 60)

# Check if already on local disk
if Path(LOCAL_DATASET_PATH).exists():
    print(f"Dataset already on local disk: {LOCAL_DATASET_PATH}")

    # Verify structure
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

# Copy and extract from Drive
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found compressed dataset on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying compressed file to local disk...")

    # Copy compressed file with progress
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} /dev/shm/sunrgbd_19_traintest.tar.gz

    # Extract to local disk
    print(f"\nExtracting dataset to local disk...")
    !tar -xzf /dev/shm/sunrgbd_19_traintest.tar.gz -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    # Remove tar file to save space
    !rm /dev/shm/sunrgbd_19_traintest.tar.gz

    print(f"\nDataset extracted to local disk")

    # Verify extraction
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected location: {DRIVE_DATASET_TAR}")
    raise FileNotFoundError(f"Compressed dataset not found at {DRIVE_DATASET_TAR}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)

SUN RGB-D 15-CATEGORY DATASET SETUP (TRAIN + TEST)
Found compressed dataset on Drive: /content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz
Copying compressed file to local disk...
          1.54G 100%   32.27MB/s    0:00:45 (xfr#1, to-chk=0/1)

Extracting dataset to local disk...

Dataset extracted to local disk

Dataset ready at: /dev/shm/sunrgbd_19_traintest


## 6. Setup Python Path & Imports


In [7]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify project structure
print('Project structure:')
!ls -la {project_root}/src/models/

# Import dataloaders and base resnet
print('\nImporting dataloaders and ResNet...')
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders
from src.training.augmentation_config import AugmentationConfig
from src.models.core.resnet import resnet18

print('All imports successful!')


Project structure:
total 48
drwxr-xr-x 11 root root 4096 Mar 27 14:34 .
drwxr-xr-x  7 root root 4096 Mar 27 14:34 ..
drwxr-xr-x  2 root root 4096 Mar 27 14:34 abstracts
drwxr-xr-x  2 root root 4096 Mar 27 14:34 common
drwxr-xr-x  2 root root 4096 Mar 27 14:34 core
drwxr-xr-x  2 root root 4096 Mar 27 14:34 direct_mixing_activation
drwxr-xr-x  2 root root 4096 Mar 27 14:34 direct_mixing_bn
drwxr-xr-x  2 root root 4096 Mar 27 14:34 direct_mixing_conv
-rw-r--r--  1 root root 1076 Mar 27 14:34 __init__.py
drwxr-xr-x  4 root root 4096 Mar 27 14:34 linear_integration
drwxr-xr-x  2 root root 4096 Mar 27 14:34 multi_channel
drwxr-xr-x  2 root root 4096 Mar 27 14:34 utils

Importing LiNet, dataloaders, and visualization tools...
All imports successful!


In [ ]:
# Set random seed for reproducibility
from src.utils.seed import set_seed

SEED = 152
DETERMINISTIC = False  # False = faster, True = fully reproducible

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

Seed: 42, Deterministic: False


## 7. Configuration

All hyperparameters and settings in one place. Modify these before running.

In [ ]:
from src.training.augmentation_config import AugmentationConfig

# ======================== DATASET ========================
DATASET_CONFIG = {
    'data_root': LOCAL_DATASET_PATH,
    'batch_size': 64,
    'num_workers': 5,
    'num_classes': 19,
    'seed': SEED
}

AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=1.0361,
    rgb_aug_mag=0.9495,
    depth_aug_prob=1.0156,
    depth_aug_mag=0.8375,
)

# ======================== MODEL ========================
MODEL_CONFIG = {
    'architecture': 'resnet18',
    'num_classes': 19,
    'width_multiplier': 0.75,
    'dropout_p': 0.4089,
    'device': 'cuda',
    'use_amp': True
}

# ======================== OPTIMIZER ========================
OPTIMIZER_CONFIG = {
    'lr': 1.01e-04,
    'weight_decay': 2.16e-04,
}

SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 110,
    'eta_min': 1.26e-06,
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2
}

# ======================== TRAINING ========================
TRAIN_CONFIG = {
    'epochs': 115,
    'grad_clip_norm': 1.1846,
    'label_smoothing': 0.1336,
}

# Print summary
print('All configs defined.')
print(f'  Dataset: {DATASET_CONFIG["data_root"]}')
print(f'  Model: ResNet18 (0.75x width)')
print(f'  LR: {OPTIMIZER_CONFIG["lr"]}, WD: {OPTIMIZER_CONFIG["weight_decay"]}')
print(f'  Epochs: {TRAIN_CONFIG["epochs"]}, Grad clip: {TRAIN_CONFIG["grad_clip_norm"]}')
print(f'  Label smoothing: {TRAIN_CONFIG["label_smoothing"]}')


All configs defined.
  Dataset: /dev/shm/sunrgbd_19_traintest
  Model: LINet3-resnet18 (2-stream)
  Streams: {0: 'RGB', 1: 'Depth'}
  Epochs: 110, Grad clip: 0.1
  Gradient monitoring: True
  Integration weight tracking: True


## 8. Load Dataset

In [10]:
# Verify dataset structure
from pathlib import Path

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_DATASET_PATH)

print("\nDirectory structure:")
print(f"  {dataset_root}/")
for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        print(f"    {split}/")
        for modality in ['rgb', 'depth']:
            mod_dir = split_dir / modality
            if mod_dir.exists():
                print(f"      {modality}/ - {len(list(mod_dir.glob('*.png')))} images")
        print(f"      labels.txt")

# Read class names
class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = [line.strip() for line in f]
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")

print("\n" + "=" * 60)

DATASET STRUCTURE VERIFICATION

Directory structure:
  /dev/shm/sunrgbd_19_traintest/
    train/
      labels.txt
    test/
      labels.txt

Classes (19):
  0: 0: bathroom
  1: 1: bedroom
  2: 2: classroom
  3: 3: computer_room
  4: 4: conference_room
  5: 5: corridor
  6: 6: dining_area
  7: 7: dining_room
  8: 8: discussion_area
  9: 9: furniture_store
  10: 10: home_office
  11: 11: kitchen
  12: 12: lab
  13: 13: lecture_theatre
  14: 14: library
  15: 15: living_room
  16: 16: office
  17: 17: rest_space
  18: 18: study_space



In [ ]:
print("=" * 60)
print("LOADING SUN RGB-D 19-CATEGORY DATASET (TRAIN + TEST)")
print("=" * 60)

print(f"\nLoading dataset from: {DATASET_CONFIG['data_root']}")

# Create dataloaders (val_loader will be None since no val/ directory)
train_loader, val_loader, test_loader = get_sunrgbd_dataloaders(
    data_root=DATASET_CONFIG['data_root'],
    batch_size=DATASET_CONFIG['batch_size'],
    num_workers=DATASET_CONFIG['num_workers'],
    seed=DATASET_CONFIG['seed'],
    **AUGMENTATION_CONFIG.to_dict(),
    stratified=True,
    normalize=True
)

print(f"\nDataset loaded!")
print(f"  Train: {len(train_loader.dataset)} samples ({len(train_loader)} batches)")
print(f"  Test: {len(test_loader.dataset)} samples ({len(test_loader)} batches)")
print(f"  Val: {'None (no val split)' if val_loader is None else f'{len(val_loader.dataset)} samples'}")

# Test loading a batch
# rgb_batch, depth_batch, label_batch = next(iter(train_loader))
# print(f"\nBatch shapes: RGB={rgb_batch.shape}, Depth={depth_batch.shape}, Labels={label_batch.shape}")

print("\n" + "=" * 60)

LOADING SUN RGB-D 19-CATEGORY DATASET (TRAIN + TEST)

Loading dataset from: /dev/shm/sunrgbd_19_traintest
Loaded SUN RGB-D train: 4845 samples, 19 classes (tensors, mmap)

Augmentation scaling applied:
  RGB:   prob=1.00, mag=1.20
  Depth: prob=1.00, mag=1.08
  Computed values:
    [Sync]  Flip prob: 0.50 -> 0.500
    [RGB]   ColorJitter prob: 0.43 -> 0.430
    [RGB]   Brightness: ±0.37 -> ±0.444
    [RGB]   Blur prob: 0.25 -> 0.250
    [RGB]   Grayscale prob: 0.17 -> 0.170
    [RGB]   Erasing prob: 0.17 -> 0.170
    [Depth] Aug prob: 0.50 -> 0.500
    [Depth] Brightness: ±0.25 -> ±0.270
    [Depth] Noise std: 0.059 -> 0.064
    [Depth] Erasing prob: 0.10 -> 0.100
Loaded SUN RGB-D test: 4659 samples, 19 classes (tensors, mmap)

Stratified sampling enabled (training only):
  Train class imbalance: 14.6x
  Each training batch will have balanced class representation

DataLoader Info:
  Train batches: 76
  Val batches: N/A (no val split)
  Test batches: 73
  Batch size: 64
  Stratified: Tr

## 8b. Baselines: ResNet18 (0.75x width)

**Four baselines:**
1. **RGB-only:** Standard ResNet18 with 3-channel RGB input
2. **Depth-only:** ResNet18 with 1-channel depth input
3. **Early fusion:** Concatenate RGB (3ch) + Depth (1ch) = 4-channel input into a single ResNet18
4. **Late fusion:** Two separate ResNet18 backbones (one RGB, one Depth), features concatenated before a shared classification head

All use identical training config (optimizer, scheduler, grad clipping, label smoothing, dropout, epochs).


In [12]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from src.models.core.resnet import resnet18

# --- Baseline training config (matches LINet where applicable) ---
BASELINE_CONFIG = {
    'epochs': TRAIN_CONFIG['epochs'],
    'grad_clip_norm': TRAIN_CONFIG['grad_clip_norm'],
    'label_smoothing': TRAIN_CONFIG['label_smoothing'],
    'dropout_p': MODEL_CONFIG['dropout_p'],
    'width_multiplier': MODEL_CONFIG['width_multiplier'],
    'lr': OPTIMIZER_CONFIG['lr'],
    'weight_decay': OPTIMIZER_CONFIG['weight_decay'],
}

print('Baseline training config:')
for k, v in BASELINE_CONFIG.items():
    print(f'  {k}: {v}')
print(f"  Scheduler: cosine, t_max={SCHEDULER_CONFIG['t_max']}, eta_min={SCHEDULER_CONFIG['eta_min']}")
print(f"  Warmup: {SCHEDULER_CONFIG['warmup_epochs']} epochs, start_factor={SCHEDULER_CONFIG['warmup_start_factor']}")

# --- Wrapper datasets (preserves dynamic augmentation from underlying dataset) ---
class RGBOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (rgb, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return rgb, label

class DepthOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (depth, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return depth, label

class EarlyFusionDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (cat(rgb, depth), label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return torch.cat([rgb, depth], dim=0), label

# --- Single-input training function ---
def train_baseline(model, train_dl, test_dl, tag=''):
    """Train a single-input baseline."""
    device = MODEL_CONFIG['device']
    use_amp = MODEL_CONFIG['use_amp']
    epochs = BASELINE_CONFIG['epochs']

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=BASELINE_CONFIG['lr'],
                                  weight_decay=BASELINE_CONFIG['weight_decay'])

    warmup = LinearLR(
        optimizer,
        start_factor=SCHEDULER_CONFIG['warmup_start_factor'],
        end_factor=1.0,
        total_iters=SCHEDULER_CONFIG['warmup_epochs']
    )
    cosine = CosineAnnealingLR(
        optimizer,
        T_max=SCHEDULER_CONFIG['t_max'] - SCHEDULER_CONFIG['warmup_epochs'],
        eta_min=SCHEDULER_CONFIG['eta_min']
    )
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[SCHEDULER_CONFIG['warmup_epochs']]
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=BASELINE_CONFIG['label_smoothing'])
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    history = {'train_loss': [], 'train_acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_dl:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(inputs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), BASELINE_CONFIG['grad_clip_norm'])
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * labels.size(0)
            correct += logits.argmax(1).eq(labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        train_loss = total_loss / total
        train_acc = correct / total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  [{tag}] E{epoch+1:3d}/{epochs} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | lr={lr_now:.2e}')

    # Single test evaluation at the end
    model.eval()
    num_classes = MODEL_CONFIG['num_classes']
    t_loss, t_correct, t_total = 0.0, 0, 0
    t_all_preds, t_all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_dl:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(inputs)
                loss = criterion(logits, labels)
            t_loss += loss.item() * labels.size(0)
            preds = logits.argmax(1)
            t_correct += preds.eq(labels).sum().item()
            t_total += labels.size(0)
            t_all_preds.append(preds.cpu())
            t_all_labels.append(labels.cpu())
    history['test_loss'] = t_loss / t_total
    history['test_acc'] = t_correct / t_total
    # Mean class accuracy
    t_all_preds = torch.cat(t_all_preds)
    t_all_labels = torch.cat(t_all_labels)
    per_class = [(t_all_preds[t_all_labels == c] == c).float().mean().item()
                 for c in range(num_classes) if (t_all_labels == c).sum() > 0]
    history['test_mca'] = sum(per_class) / len(per_class)

    return history


Baseline training config:
  epochs: 110
  grad_clip_norm: 0.1
  label_smoothing: 0.119
  dropout_p: 0.422
  width_multiplier: 0.75
  Scheduler: cosine, t_max=105, eta_min=5e-07
  Warmup: 5 epochs, start_factor=0.2


In [ ]:
# --- RGB-Only: standard 3-channel ResNet18 ---
print('=' * 60)
print('BASELINE: ResNet18 0.75x width — RGB Only')
print('=' * 60)

from thop import profile

rgb_only = resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=BASELINE_CONFIG['width_multiplier'],
    device='cpu',
    use_amp=False
)

# Add dropout before FC
rgb_only.fc = nn.Sequential(
    nn.Dropout(p=BASELINE_CONFIG['dropout_p']),
    nn.Linear(rgb_only.fc.in_features, rgb_only.fc.out_features)
)
rgb_only.to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in rgb_only.parameters())
print(f'Parameters: {n_params:,}')

dummy = torch.randn(1, 3, 224, 224).to(MODEL_CONFIG['device'])
flops, _ = profile(rgb_only, inputs=(dummy,), verbose=False)
print(f'GFLOPs: {flops / 1e9:.3f}')
del dummy

# Build RGB-only DataLoaders
bs = train_loader.batch_size
nw = train_loader.num_workers
rgb_train_dl = DataLoader(RGBOnlyDataset(train_loader.dataset),
                          batch_size=bs, shuffle=True, num_workers=nw, pin_memory=True)
rgb_test_dl = DataLoader(RGBOnlyDataset(test_loader.dataset),
                         batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)

rgb_history = train_baseline(rgb_only, rgb_train_dl, rgb_test_dl, tag='RGB-Only')
print(f"\nRGB-Only test accuracy: {rgb_history['test_acc']:.4f}  MCA: {rgb_history['test_mca']:.4f}")


BASELINE: ResNet18 0.75x width — Early Fusion (RGB+Depth concat)
Parameters: 6,300,019
Early fusion loaders: train=4845, test=4659, shape=torch.Size([4, 224, 224])
  [EarlyFusion] E  1/110 | train_loss=2.8238 train_acc=0.1536 | lr=4.84e-05
  [EarlyFusion] E 10/110 | train_loss=2.1224 train_acc=0.4239 | lr=1.34e-04
  [EarlyFusion] E 20/110 | train_loss=1.7985 train_acc=0.5544 | lr=1.27e-04
  [EarlyFusion] E 30/110 | train_loss=1.5873 train_acc=0.6409 | lr=1.15e-04
  [EarlyFusion] E 40/110 | train_loss=1.4113 train_acc=0.7203 | lr=9.80e-05
  [EarlyFusion] E 50/110 | train_loss=1.2389 train_acc=0.8004 | lr=7.80e-05
  [EarlyFusion] E 60/110 | train_loss=1.1238 train_acc=0.8487 | lr=5.70e-05
  [EarlyFusion] E 70/110 | train_loss=1.0314 train_acc=0.8937 | lr=3.71e-05
  [EarlyFusion] E 80/110 | train_loss=0.9788 train_acc=0.9216 | lr=2.01e-05
  [EarlyFusion] E 90/110 | train_loss=0.9432 train_acc=0.9370 | lr=7.81e-06
  [EarlyFusion] E100/110 | train_loss=0.9270 train_acc=0.9455 | lr=1.33e-06


In [ ]:
# Visualize learned conv1 filters — RGB-Only
import math
from torchvision.utils import make_grid
w = rgb_only.conv1.weight.detach().cpu()
nrow = int(math.ceil(math.sqrt(w.shape[0])))
grid = make_grid(w, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title(f'RGB-Only — {w.shape[0]} filters ({w.shape[1]}x{w.shape[2]}x{w.shape[3]})', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# --- Depth-Only: 1-channel input ResNet18 ---
print('=' * 60)
print('BASELINE: ResNet18 0.75x width — Depth Only')
print('=' * 60)

depth_only = resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=BASELINE_CONFIG['width_multiplier'],
    device='cpu',
    use_amp=False
)

# Replace conv1: 1 input channel for depth
old_conv1 = depth_only.conv1
depth_only.conv1 = nn.Conv2d(
    1, old_conv1.out_channels,
    kernel_size=7, stride=2, padding=3, bias=False
)
nn.init.kaiming_normal_(depth_only.conv1.weight, mode='fan_out', nonlinearity='relu')

# Add dropout before FC
depth_only.fc = nn.Sequential(
    nn.Dropout(p=BASELINE_CONFIG['dropout_p']),
    nn.Linear(depth_only.fc.in_features, depth_only.fc.out_features)
)
depth_only.to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in depth_only.parameters())
print(f'Parameters: {n_params:,}')

dummy = torch.randn(1, 1, 224, 224).to(MODEL_CONFIG['device'])
flops, _ = profile(depth_only, inputs=(dummy,), verbose=False)
print(f'GFLOPs: {flops / 1e9:.3f}')
del dummy

# Build Depth-only DataLoaders
depth_train_dl = DataLoader(DepthOnlyDataset(train_loader.dataset),
                            batch_size=bs, shuffle=True, num_workers=nw, pin_memory=True)
depth_test_dl = DataLoader(DepthOnlyDataset(test_loader.dataset),
                           batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)

depth_history = train_baseline(depth_only, depth_train_dl, depth_test_dl, tag='Depth-Only')
print(f"\nDepth-Only test accuracy: {depth_history['test_acc']:.4f}  MCA: {depth_history['test_mca']:.4f}")


BASELINE: ResNet18 0.75x width — Late Fusion (feature concat)
Parameters: 12,590,611


In [ ]:
# Visualize learned conv1 filters — Depth-Only
import math
from torchvision.utils import make_grid
w = depth_only.conv1.weight.detach().cpu()
n_filters, in_ch, kh, kw = w.shape
w = w.repeat(1, 3, 1, 1)
nrow = int(math.ceil(math.sqrt(n_filters)))
grid = make_grid(w, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title(f'Depth-Only — {n_filters} filters ({in_ch}x{kh}x{kw})', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()


In [15]:
# --- Early Fusion: 4-channel input ResNet18 ---
print('=' * 60)
print('BASELINE: ResNet18 0.75x width — Early Fusion (RGB+Depth concat)')
print('=' * 60)

early_fusion = resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=BASELINE_CONFIG['width_multiplier'],
    device='cpu',
    use_amp=False
)

# Replace conv1: 4 input channels (RGB=3 + Depth=1)
old_conv1 = early_fusion.conv1
early_fusion.conv1 = nn.Conv2d(
    4, old_conv1.out_channels,
    kernel_size=7, stride=2, padding=3, bias=False
)
nn.init.kaiming_normal_(early_fusion.conv1.weight, mode='fan_out', nonlinearity='relu')

# Add dropout before FC
early_fusion.fc = nn.Sequential(
    nn.Dropout(p=BASELINE_CONFIG['dropout_p']),
    nn.Linear(early_fusion.fc.in_features, early_fusion.fc.out_features)
)
early_fusion.to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in early_fusion.parameters())
print(f'Parameters: {n_params:,}')

dummy = torch.randn(1, 4, 224, 224).to(MODEL_CONFIG['device'])
flops, _ = profile(early_fusion, inputs=(dummy,), verbose=False)
print(f'GFLOPs: {flops / 1e9:.3f}')
del dummy

# Build early-fusion DataLoaders
ef_train_dl = DataLoader(EarlyFusionDataset(train_loader.dataset),
                         batch_size=bs, shuffle=True, num_workers=nw, pin_memory=True)
ef_test_dl = DataLoader(EarlyFusionDataset(test_loader.dataset),
                        batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)

ef_history = train_baseline(early_fusion, ef_train_dl, ef_test_dl, tag='EarlyFusion')
print(f"\nEarly Fusion test accuracy: {ef_history['test_acc']:.4f}  MCA: {ef_history['test_mca']:.4f}")


  [LateFusion] E  1/110 | train_loss=2.9465 train_acc=0.1024 | lr=3.57e-05
  [LateFusion] E 10/110 | train_loss=2.0776 train_acc=0.4351 | lr=9.85e-05
  [LateFusion] E 20/110 | train_loss=1.6618 train_acc=0.6194 | lr=9.37e-05
  [LateFusion] E 30/110 | train_loss=1.4022 train_acc=0.7389 | lr=8.47e-05
  [LateFusion] E 40/110 | train_loss=1.2214 train_acc=0.8169 | lr=7.22e-05
  [LateFusion] E 50/110 | train_loss=1.0866 train_acc=0.8813 | lr=5.75e-05
  [LateFusion] E 60/110 | train_loss=0.9839 train_acc=0.9302 | lr=4.21e-05
  [LateFusion] E 70/110 | train_loss=0.9374 train_acc=0.9459 | lr=2.74e-05
  [LateFusion] E 80/110 | train_loss=0.8990 train_acc=0.9602 | lr=1.49e-05
  [LateFusion] E 90/110 | train_loss=0.8667 train_acc=0.9730 | lr=5.87e-06
  [LateFusion] E100/110 | train_loss=0.8620 train_acc=0.9763 | lr=1.11e-06
  [LateFusion] E110/110 | train_loss=0.8523 train_acc=0.9792 | lr=1.11e-06

Late Fusion final test accuracy: 0.4866


In [ ]:
# Visualize learned conv1 filters — Early Fusion (4ch: RGB+Depth)
import math
from torchvision.utils import make_grid
w = early_fusion.conv1.weight.detach().cpu()
nrow = int(math.ceil(math.sqrt(w.shape[0])))
fig, axes = plt.subplots(2, 1, figsize=(8, 16))
# RGB channels (first 3)
grid_rgb = make_grid(w[:, :3], nrow=nrow, normalize=True, scale_each=True, pad_value=1)
axes[0].imshow(grid_rgb.permute(1, 2, 0).numpy())
axes[0].set_title(f'RGB Channels — {w.shape[0]} filters (3x{w.shape[2]}x{w.shape[3]})', fontsize=14)
axes[0].axis('off')
# Depth channel (4th)
w_d = w[:, 3:4].repeat(1, 3, 1, 1)
grid_d = make_grid(w_d, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
axes[1].imshow(grid_d.permute(1, 2, 0).numpy())
axes[1].set_title(f'Depth Channel — {w.shape[0]} filters (1x{w.shape[2]}x{w.shape[3]})', fontsize=14)
axes[1].axis('off')
fig.suptitle('Early Fusion — Learned First-Layer Filters', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# --- Late Fusion: two ResNet18 backbones + shared head ---
print('=' * 60)
print('BASELINE: ResNet18 0.75x width — Late Fusion (feature concat)')
print('=' * 60)

class LateFusionResNet(nn.Module):
    def __init__(self, num_classes, width_multiplier, dropout_p):
        super().__init__()
        self.rgb_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False
        )
        self.depth_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False
        )
        self.depth_backbone.conv1 = nn.Conv2d(
            1, self.depth_backbone.conv1.out_channels,
            kernel_size=7, stride=2, padding=3, bias=False
        )
        nn.init.kaiming_normal_(self.depth_backbone.conv1.weight, mode='fan_out', nonlinearity='relu')

        feat_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()
        self.depth_backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(feat_dim * 2, num_classes)
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)
        fused = torch.cat([rgb_feat, depth_feat], dim=1)
        return self.classifier(fused)

late_fusion = LateFusionResNet(
    num_classes=MODEL_CONFIG['num_classes'],
    width_multiplier=BASELINE_CONFIG['width_multiplier'],
    dropout_p=BASELINE_CONFIG['dropout_p'],
).to(MODEL_CONFIG['device'])

n_params = sum(p.numel() for p in late_fusion.parameters())
print(f'Parameters: {n_params:,}')

dummy_rgb = torch.randn(1, 3, 224, 224).to(MODEL_CONFIG['device'])
dummy_depth = torch.randn(1, 1, 224, 224).to(MODEL_CONFIG['device'])
lf_flops, _ = profile(late_fusion, inputs=(dummy_rgb, dummy_depth), verbose=False)
print(f'GFLOPs: {lf_flops / 1e9:.3f}')
del dummy_rgb, dummy_depth

# Train late fusion (2-input forward, but single optimizer param group)
device = MODEL_CONFIG['device']
use_amp = MODEL_CONFIG['use_amp']
epochs = BASELINE_CONFIG['epochs']

optimizer = torch.optim.AdamW(late_fusion.parameters(),
                              lr=BASELINE_CONFIG['lr'],
                              weight_decay=BASELINE_CONFIG['weight_decay'])

warmup = LinearLR(
    optimizer,
    start_factor=SCHEDULER_CONFIG['warmup_start_factor'],
    end_factor=1.0,
    total_iters=SCHEDULER_CONFIG['warmup_epochs']
)
cosine = CosineAnnealingLR(
    optimizer,
    T_max=SCHEDULER_CONFIG['t_max'] - SCHEDULER_CONFIG['warmup_epochs'],
    eta_min=SCHEDULER_CONFIG['eta_min']
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup, cosine],
    milestones=[SCHEDULER_CONFIG['warmup_epochs']]
)

criterion = nn.CrossEntropyLoss(label_smoothing=BASELINE_CONFIG['label_smoothing'])
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

lf_history = {'train_loss': [], 'train_acc': []}

for epoch in range(epochs):
    late_fusion.train()
    total_loss, correct, total = 0.0, 0, 0
    for rgb, depth, labels in train_loader:
        rgb = rgb.to(device, non_blocking=True)
        depth = depth.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = late_fusion(rgb, depth)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(late_fusion.parameters(), BASELINE_CONFIG['grad_clip_norm'])
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        correct += logits.argmax(1).eq(labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = total_loss / total
    train_acc = correct / total
    lf_history['train_loss'].append(train_loss)
    lf_history['train_acc'].append(train_acc)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  [LateFusion] E{epoch+1:3d}/{epochs} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | lr={lr_now:.2e}')

# Test evaluation
late_fusion.eval()
num_classes = MODEL_CONFIG['num_classes']
t_loss, t_correct, t_total = 0.0, 0, 0
t_all_preds, t_all_labels = [], []
with torch.no_grad():
    for rgb, depth, labels in test_loader:
        rgb = rgb.to(device, non_blocking=True)
        depth = depth.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = late_fusion(rgb, depth)
            loss = criterion(logits, labels)
        t_loss += loss.item() * labels.size(0)
        preds = logits.argmax(1)
        t_correct += preds.eq(labels).sum().item()
        t_total += labels.size(0)
        t_all_preds.append(preds.cpu())
        t_all_labels.append(labels.cpu())
lf_history['test_loss'] = t_loss / t_total
lf_history['test_acc'] = t_correct / t_total
t_all_preds = torch.cat(t_all_preds)
t_all_labels = torch.cat(t_all_labels)
per_class = [(t_all_preds[t_all_labels == c] == c).float().mean().item()
             for c in range(num_classes) if (t_all_labels == c).sum() > 0]
lf_history['test_mca'] = sum(per_class) / len(per_class)

print(f"\nLate Fusion test accuracy: {lf_history['test_acc']:.4f}  MCA: {lf_history['test_mca']:.4f}")


In [ ]:
# Visualize learned conv1 filters — Late Fusion (RGB + Depth backbones)
import math
from torchvision.utils import make_grid
fig, axes = plt.subplots(2, 1, figsize=(8, 16))
for si, (backbone, label) in enumerate([
    (late_fusion.rgb_backbone, 'RGB Backbone'),
    (late_fusion.depth_backbone, 'Depth Backbone'),
]):
    w = backbone.conv1.weight.detach().cpu()
    n_filters, in_ch, kh, kw = w.shape
    if in_ch == 1:
        w = w.repeat(1, 3, 1, 1)
    nrow = int(math.ceil(math.sqrt(n_filters)))
    grid = make_grid(w, nrow=nrow, normalize=True, scale_each=True, pad_value=1)
    axes[si].imshow(grid.permute(1, 2, 0).numpy())
    axes[si].set_title(f'{label} — {n_filters} filters ({in_ch}x{kh}x{kw})', fontsize=14)
    axes[si].axis('off')
fig.suptitle('Late Fusion — Learned First-Layer Filters', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Training Curves


In [ ]:
# --- Training curves: all four baselines ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_x = range(1, BASELINE_CONFIG['epochs'] + 1)
colors = {'RGB-Only': 'tab:blue', 'Depth-Only': 'tab:orange',
          'Early Fusion': 'tab:green', 'Late Fusion': 'tab:purple'}
histories = {
    'RGB-Only': rgb_history,
    'Depth-Only': depth_history,
    'Early Fusion': ef_history,
    'Late Fusion': lf_history,
}

for name, h in histories.items():
    c = colors[name]
    axes[0].plot(epochs_x, h['train_loss'], label=f'{name} train', color=c)
    axes[0].axhline(y=h['test_loss'], color=c, linestyle='--', alpha=0.7,
                    label=f"{name} test ({h['test_loss']:.3f})")
    axes[1].plot(epochs_x, h['train_acc'], label=f'{name} train', color=c)
    axes[1].axhline(y=h['test_acc'], color=c, linestyle='--', alpha=0.7,
                    label=f"{name} test ({h['test_acc']:.3f})")

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'ResNet18 (0.75x) Baselines — {BASELINE_CONFIG["epochs"]} epochs', fontsize=13)
plt.tight_layout()
plt.show()


## 10. Baseline Results Summary


In [ ]:
# --- Summary table ---
print('=' * 70)
print('BASELINE RESULTS — ResNet18 (0.75x width) on SUN RGB-D 19-Category')
print('=' * 70)
print(f'{"Model":<20} {"Test Acc":>10} {"Test MCA":>10} {"Test Loss":>10}')
print('-' * 50)

results = {
    'RGB-Only': rgb_history,
    'Depth-Only': depth_history,
    'Early Fusion': ef_history,
    'Late Fusion': lf_history,
}

for name, h in results.items():
    print(f"{name:<20} {h['test_acc']:>10.4f} {h['test_mca']:>10.4f} {h['test_loss']:>10.4f}")

print('=' * 70)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
mcas = [results[n]['test_mca'] for n in names]
accs = [results[n]['test_acc'] for n in names]

x = range(len(names))
width = 0.35
ax.bar([i - width/2 for i in x], accs, width, label='Test Accuracy', color='steelblue')
ax.bar([i + width/2 for i in x], mcas, width, label='Mean Class Accuracy', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(names)
ax.set_ylabel('Score')
ax.set_title('Baseline Comparison — SUN RGB-D 19-Category')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1)

for i, (a, m) in enumerate(zip(accs, mcas)):
    ax.text(i - width/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, m + 0.01, f'{m:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()
